# Orca Core (Novus) — Calibration Fix SFT (Kaggle, on top of DPO v1, plain PEFT)

**Why this notebook exists**: `orca train redteam --model orca-core-dpo --jailbreak-trials 3 --bias-trials 3` measured a real 16.7% calibration score (5/6 false-premise probes missed) — the model confidently builds an answer on top of a false premise (e.g. "since goldfish have 3-second memory...") instead of catching and correcting it. This is a distinct skill gap from jailbreak resistance, not something the safety DPO round touched or was meant to fix.

**Why SFT, not DPO**: this is a missing *skill* (catch-and-correct a false premise), not a preference between two responses to elicit — plain demonstration data is the right tool, same reasoning `orca/train/dpo_pairs.py`'s own docstring gives for when DPO is/isn't appropriate.

**Why plain PEFT, not Unsloth**: 4 straight runs of an Unsloth-based version of this notebook failed — a P100 GPU assignment, a multi-GPU attention crash, a silently-ignored `attn_implementation="sdpa"` kwarg (Unsloth patches attention internally regardless), and finally a broken `xformers<0.0.27` pip build (no prebuilt wheel for Kaggle's current CUDA/torch stack). All four traced back to the same root cause: Unsloth's patched attention forces xformers' `memory_efficient_attention`, which has no working backend for Llama-3.1's grouped-query-attention tensor shape on T4 right now. This dataset is tiny (54 examples, 12 training steps) — Unsloth's speed/memory optimizations buy nothing here, so this switches to plain `transformers` + `peft` + `bitsandbytes`, the same lesson already applied to the merge/export step in `orca_core_dpo_merge_export_v1.ipynb` after Unsloth's convenience wrappers proved fragile there too. Plain `AutoModelForCausalLM` genuinely respects `attn_implementation="sdpa"` and never touches xformers.

**Data**: 60 real examples (54 train / 6 eval) generated via a new `premise_correction` domain in `orca/data/seeds.py`, using `llama3.1:8b` as a free local teacher (0 failures across all 60). Deliberately covers **different** myths than the 6 fixed eval probes in `orca/train/redteam.py`'s `CALIBRATION_PROBES` — training on the literal eval probes would teach memorization, not the general skill the eval is trying to measure.

**Continuing from DPO, not from scratch**: loads the base model + attaches the round-1 `adapter_dpo` LoRA (via `dataset_sources`: `orca-core-dpo-adapter-v1`) as a trainable PEFT model, so this SFT pass builds on the jailbreak-resistance gains already made instead of undoing them.


In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "0"

# Plain transformers/peft/bitsandbytes stack -- no unsloth, no xformers, no trl
# (trl's SFTTrainer has a real internal bug -- huggingface/trl#6483 -- that
# crashes when device_map="auto" + 4-bit loading has already wrapped the
# model's .forward via accelerate hooks; plain transformers.Trainer sidesteps
# it entirely, see the training cell below).
#
# transformers pinned <5 -- same real bug hit on the DPO round: unpinned
# installs pull transformers 5.x, whose new internal weight-conversion
# registry breaks downstream merge tooling for this architecture.
#
# torchao pinned >=0.16.0 -- same real bug hit on orca_core_dpo_merge_export_v1.ipynb:
# peft's LoRA-merge dispatch checks torchao's version and raises ImportError
# on Kaggle's default (older) preinstalled torchao during the merge step.
!pip install -q "transformers<5" "torchao>=0.16.0" peft bitsandbytes accelerate datasets

## Find the uploaded calibration dataset and the round-1 DPO adapter

In [ ]:
import glob, os

train_matches = glob.glob('/kaggle/input/**/orca_core_calibration_train_v1.jsonl', recursive=True)
eval_matches  = glob.glob('/kaggle/input/**/orca_core_calibration_eval_v1.jsonl', recursive=True)
adapter_matches = glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)

print('Train file:', train_matches)
print('Eval file:', eval_matches)
print('Adapter config:', adapter_matches)

if not train_matches or not adapter_matches:
    raise FileNotFoundError(
        "Need both the calibration train/eval dataset attached (dataset_sources: "
        "orca-core-calibration-sft-v1) and the round-1 DPO adapter attached "
        "(dataset_sources: orca-core-dpo-adapter-v1) — check the notebook's "
        "kernel-metadata.json."
    )

train_path = train_matches[0]
eval_path = eval_matches[0] if eval_matches else None
adapter_dir = os.path.dirname(adapter_matches[0])
print('Using adapter_dir:', adapter_dir)

In [ ]:
import json

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except Exception:
                    pass
    return records

raw_train = load_jsonl(train_path)
raw_eval  = load_jsonl(eval_path) if eval_path else raw_train[:max(1, len(raw_train)//10)]
print(f'train={len(raw_train)} eval={len(raw_eval)}')

## Load base model (4-bit, plain bitsandbytes) + attach the round-1 DPO adapter (continuing from it, not from scratch)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training

max_seq_length = 2048
base_model_name = "unsloth/Meta-Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
)
tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = prepare_model_for_kbit_training(model)

# Attach the round-1 DPO adapter, trainable — this SFT pass continues from
# the jailbreak-resistance gains already trained in, instead of starting
# from the plain v2 SFT checkpoint (which would discard that DPO round).
model = PeftModel.from_pretrained(model, adapter_dir, is_trainable=True)
model.print_trainable_parameters()

In [ ]:
from datasets import Dataset

def _tokenize(example):
    return tokenizer(example["text"], truncation=True, max_length=max_seq_length)

train_ds = Dataset.from_list([{"text": ex["text"]} for ex in raw_train]).map(_tokenize, remove_columns=["text"])
eval_ds  = Dataset.from_list([{"text": ex["text"]} for ex in raw_eval]).map(_tokenize, remove_columns=["text"])
print(f'train_ds={len(train_ds)} eval_ds={len(eval_ds)}')

## Train

Small dataset (54 train examples) — 3 epochs, same effective-batch-16 sizing
as the original core SFT run. Plain `transformers.Trainer`, not trl's
`SFTTrainer` — trl's newer "chunked cross-entropy on the LM head" memory
optimization has a real bug (huggingface/trl#6483) that crashes whenever the
model's `.forward` has already been wrapped by accelerate's device-dispatch
hooks (which happens with `device_map="auto"` + 4-bit loading regardless of
gradient checkpointing — confirmed by testing both with and without it).
Since our data is already pre-formatted as full chat-template text per
example (no packing needed), a plain `Trainer` does everything `SFTTrainer`
would here, without going anywhere near that buggy code path. No mid-training
checkpointing (`save_strategy="no"`) — the adapter-save cell right after
training is the real safety net, per the same real Kaggle-side pickling bug
hit on earlier runs.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
import time

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=1e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=2,
    eval_steps=5,
    save_strategy="no",
    output_dir="/kaggle/working/output",
    eval_strategy="steps",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
)

print("[train] starting calibration-fix QLoRA training (plain PEFT + plain Trainer, no unsloth, no trl)...")
t0 = time.time()
trainer.train()
elapsed = (time.time() - t0) / 60
print(f"[train] done in {elapsed:.1f} min")

## Save the updated adapter immediately (before merge/export)

In [ ]:
adapter_out_dir = "/kaggle/working/adapter_calibration"
model.save_pretrained(adapter_out_dir)
tokenizer.save_pretrained(adapter_out_dir)
print(f"[adapter] saved to {adapter_out_dir} — calibration-corrected weights are now safe on disk.")
!ls -la {adapter_out_dir}

## Merge LoRA + export GGUF — plain PEFT, base model reloaded in fp16 (never 4-bit)

Same approach as `orca_core_dpo_merge_export_v1.ipynb` (proven to work): the
training model above stays 4-bit for memory reasons, but merging a 4-bit
model directly is fragile (packed tensors, dequant edge cases). Reloading
the base fresh in plain fp16 means there's no quantization config anywhere
in this path — nothing stale to strip, nothing to dequantize.

In [ ]:
import gc, torch as _torch

# Free the 4-bit training model before loading a second fp16 copy — T4 has
# 16GB VRAM, not enough for both at once.
del model, trainer
gc.collect()
_torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM as _AutoModelForCausalLM
from peft import PeftModel as _PeftModel

print("[load] loading base model in fp16 for merge...")
base_fp16 = _AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=_torch.float16,
    device_map="auto",
)

print("[merge] attaching the just-trained adapter and merging...")
merge_model = _PeftModel.from_pretrained(base_fp16, adapter_out_dir)
merged = merge_model.merge_and_unload()
print("[merge] done — merged is a plain AutoModelForCausalLM, no PEFT wrapper, no quant config")

assert getattr(merged.config, "quantization_config", None) is None, (
    "merged model still has a quantization_config — something upstream "
    "changed and this notebook's core assumption no longer holds, stop here."
)
print("[check] confirmed: no quantization_config on the merged model")

import shutil
shutil.rmtree("/tmp/merged_clean", ignore_errors=True)
merged.save_pretrained("/tmp/merged_clean", safe_serialization=True)
tokenizer.save_pretrained("/tmp/merged_clean")
print("[save] merged model saved to /tmp/merged_clean")
!ls -la /tmp/merged_clean

In [ ]:
## Convert to GGUF using the standard llama.cpp project directly, then quantize to Q4_K_M

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp
!pip install -q -r /tmp/llama.cpp/requirements.txt

In [ ]:
import os

os.makedirs("/tmp/gguf_out", exist_ok=True)
f16_path = "/tmp/gguf_out/orca-core-calibration.F16.gguf"

print("[convert] running llama.cpp's convert_hf_to_gguf.py...")
!python3 /tmp/llama.cpp/convert_hf_to_gguf.py /tmp/merged_clean --outfile {f16_path} --outtype f16
print("[convert] done, checking output:")
!ls -la /tmp/gguf_out

## Build llama.cpp's quantize tool and produce the final Q4_K_M GGUF

In [ ]:
!cmake -B /tmp/llama.cpp/build -S /tmp/llama.cpp -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=OFF
!cmake --build /tmp/llama.cpp/build --config Release -j --target llama-quantize

In [ ]:
import glob

quantize_bin_candidates = glob.glob("/tmp/llama.cpp/build/**/llama-quantize", recursive=True)
print("[quantize] llama-quantize binary found at:", quantize_bin_candidates)

if not quantize_bin_candidates:
    raise RuntimeError(
        "llama-quantize binary not found after build -- check the cmake build "
        "output above for the real error before assuming this notebook's logic "
        "is wrong; this is a standard llama.cpp build step, not custom code."
    )

q4_path = "/tmp/gguf_out/orca-core-calibration.Q4_K_M.gguf"
!{quantize_bin_candidates[0]} {f16_path} {q4_path} Q4_K_M
print("[quantize] done, checking output:")
!ls -la /tmp/gguf_out

## Copy the final Q4_K_M GGUF to /kaggle/working (only the final file — the F16 intermediate stays in /tmp, too big for the 19.5GB working quota)

In [ ]:
import shutil

dest = "/kaggle/working/orca-core-calibration.Q4_K_M.gguf"
shutil.copy(q4_path, dest)
print(f"[export] copied final quantized model to {dest}")
print("\nNext: click 'Save Version' -> 'Save & Run All (Commit)' at the top right.")